# Notebook 046: Avro Binary Ingestion and Schema Registry Compatibility

**Module:** Module 05: Data Ingestion Patterns, Auto Loader & File Formats  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 16: LakeFlow Connect & Video 17: Lakeflow Declarative Pipelines  

---

## 1. Business Scenario & Real-World Context
An enterprise Apache Kafka event bus publishes high-throughput customer account updates serialized in Apache Avro format. Data engineering must consume Avro payloads using explicit Avro schemas compatible with the Confluent Schema Registry.

## 2. Technical Requirement & Problem Statement
Ingest and serialize Apache Avro binary records in PySpark: use `from_avro()` and `to_avro()`, define JSON Avro schema specifications, handle schema evolution between writer and reader schemas, and package binary payloads.

## 3. Spark & Databricks Architecture Notes
Apache Avro is a row-oriented binary serialization format widely used for streaming pipelines due to its compact size and schema evolution rules. The `from_avro` function decodes binary Kafka bytes into Spark StructType columns using an Avro schema string.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_046_Avro_Binary_Ingestion_and_Schema_Registry_Compatibility")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Defining Avro Schema JSON Specification
Define standard Avro JSON schema.

In [ ]:
avro_schema_json = """{
  "type": "record",
  "name": "CustomerEvent",
  "namespace": "com.enterprise.events",
  "fields": [
    {"name": "customer_id", "type": "string"},
    {"name": "event_type", "type": "string"},
    {"name": "account_balance", "type": "double"},
    {"name": "event_timestamp", "type": "long"}
  ]
}"""

print("Avro Schema JSON defined successfully.")

## 6. Simulating Avro Ingestion and Struct Conversion
Create events and demonstrate Avro integration patterns.

In [ ]:
# Create representative structured records
events = [
    ("CUST-101", "LOGIN", 1450.50, 1700560800),
    ("CUST-102", "PURCHASE", 320.00, 1700560950),
    ("CUST-103", "LOGOUT", 8900.00, 1700561100)
]

cols = ["customer_id", "event_type", "account_balance", "event_timestamp"]
df_events = spark.createDataFrame(events, cols)
df_events.printSchema()
df_events.show()

## 7. Packaging Structs for Streaming Message Buses
Bundle fields into structs for downstream message publishing.

In [ ]:
# Convert to unified message payload struct matching Avro schema
df_payload = df_events.select(
    F.col("customer_id").alias("key"),
    F.struct("customer_id", "event_type", "account_balance", "event_timestamp").alias("value")
)

df_payload.printSchema()
df_payload.show(truncate=False)

## 9. Key Architectural Takeaways
- **Avro is the industry standard format for streaming message buses (Kafka, Event Hubs) due to compact binary encoding.**
- **`from_avro(col, schema)` parses binary payloads using the exact Avro schema definition.**
- **Schema evolution in Avro allows adding optional fields with default values without breaking downstream consumers.**

## 10. Senior Data Engineering Interview Questions
1. Why is Avro preferred for message bus streaming while Parquet is preferred for analytical storage?
2. How does Confluent Schema Registry prevent incompatible schema changes in Kafka-Spark pipelines?
3. What is the difference between Writer Schema and Reader Schema in Apache Avro?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
